# Setup

In [1]:
HIVE_START_FROM_SCRATCH = True
HIVE_VPN_DOMAIN = "lvilla17.vpn.itam.mx"
DOCKER_INTERNAL_HOST = "host.docker.internal"
DOCKER_DNS = ["10.15.20.1"]

POSTGRES_USER = "hive"
POSTGRES_PASSWORD = "hive"
POSTGRES_DB = "metastore"

HIVE_DB_CONTAINER_NAME = "hive-metastore-db"
HIVE_SCHEMA_INIT_CONTAINER_NAME = "hive-schema-init"
HIVE_METASTORE_CONTAINER_NAME = "hive-metastore"
HIVE_SERVER2_CONTAINER_NAME = "hive-server2"

HIVE_DB_HOSTNAME = f"{HIVE_DB_CONTAINER_NAME}.{HIVE_VPN_DOMAIN}"
HIVE_SCHEMA_INIT_HOSTNAME = f"{HIVE_SCHEMA_INIT_CONTAINER_NAME}.{HIVE_VPN_DOMAIN}"
HIVE_METASTORE_HOSTNAME = f"{HIVE_METASTORE_CONTAINER_NAME}.{HIVE_VPN_DOMAIN}"
HIVE_SERVER2_HOSTNAME = f"{HIVE_SERVER2_CONTAINER_NAME}.{HIVE_VPN_DOMAIN}"

HIVE_DB_INTERNAL_PORT = 15432
HIVE_METASTORE_INTERNAL_PORT = 9083
HIVE_SERVER2_INTERNAL_PORT = 10000
HIVE_SERVER2_UI_INTERNAL_PORT = 10002

HIVE_DB_EXTERNAL_PORT = 15432
HIVE_METASTORE_EXTERNAL_PORT = 9083
HIVE_SERVER2_EXTERNAL_PORT = 10000
HIVE_SERVER2_UI_EXTERNAL_PORT = 10002

HIVE_USERDIR = "/user/hive"
HIVE_DATADIR = f"{HIVE_USERDIR}/warehouse"

HADOOP_RESOURCEMANAGER_WEBUI_PORT = 8088
HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT = 8032
HADOOP_RESOURCEMANAGER_TRACKER_PORT = 8031
HADOOP_RESOURCEMANAGER_SCHEDULER_PORT = 8030
HADOOP_RESOURCEMANAGER_ADMIN_PORT = 8033

HADOOP_NAMENODE_HOSTNAME = f"namenode.{HIVE_VPN_DOMAIN}"
HADOOP_NAMENODE_PORT = 8020

HADOOP_RESOURCEMANAGER_HOSTNAME = f"resourcemanager.{HIVE_VPN_DOMAIN}"
HADOOP_RESOURCEMANAGER_PORT = 8032

APACHE_HIVE_IMAGE = "apache/hive:4.0.1"
POSTGRES_IMAGE = "postgres:18"

In [2]:
import os
from pathlib import Path

LOCAL_WORKDIR = f"{os.path.join(os.path.relpath(Path.cwd()))}"
DOCKER_MOUNTDIR = os.path.join(LOCAL_WORKDIR, "mount")

mount_path = Path(DOCKER_MOUNTDIR)
mount_path.mkdir(parents=True, exist_ok=True)

# Stop hive-cluster.docker-compose.yml

In [3]:
!docker compose -f hive-cluster.docker-compose.yml down -v

[+] down 0/1
 ⠋ Container hive-server2 Stopping                                          0.1s
[+] down 0/1
 ⠙ Container hive-server2 Stopping                                          0.2s
[+] down 0/1
 ⠹ Container hive-server2 Stopping                                          0.3s
[+] down 0/1
 ⠸ Container hive-server2 Stopping                                          0.4s
[+] down 0/1
 ⠼ Container hive-server2 Stopping                                          0.5s
[+] down 0/1
 ⠴ Container hive-server2 Stopping                                          0.6s
[+] down 0/1
 ⠦ Container hive-server2 Stopping                                          0.7s
[+] down 0/1
 ⠧ Container hive-server2 Stopping                                          0.8s
[+] down 0/1
 ⠇ Container hive-server2 Stopping                                          0.9s
[+] down 0/1
 ⠏ Container hive-server2 Stopping                                          1.0s
[+] down 0/1
 ⠋ Container hive-server2 Stopping             

In [4]:
import shutil
import stat

os.makedirs(os.path.join(DOCKER_MOUNTDIR), exist_ok=True)
if HIVE_START_FROM_SCRATCH:
    for container in [
        HIVE_DB_CONTAINER_NAME,
        HIVE_METASTORE_CONTAINER_NAME,
        HIVE_SERVER2_CONTAINER_NAME,
        HIVE_SCHEMA_INIT_CONTAINER_NAME,
    ]:
        if os.path.exists(os.path.join(DOCKER_MOUNTDIR, container)):

            def on_rm_error(func, path, exc_info):
                os.chmod(path, stat.S_IWRITE)
                os.unlink(path)

            shutil.rmtree(os.path.join(DOCKER_MOUNTDIR, container), onerror=on_rm_error)
        os.makedirs(os.path.join(DOCKER_MOUNTDIR, container), exist_ok=True)

# Create all config files

In [5]:
core_site_xml_content = f"""<?xml version="1.0"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}</value>
    </property>
</configuration>
"""

for container in [HIVE_METASTORE_CONTAINER_NAME, HIVE_SERVER2_CONTAINER_NAME]:
    dest_dir = os.path.join(DOCKER_MOUNTDIR, container, "hive_custom_conf")
    os.makedirs(dest_dir, exist_ok=True)
    with open(os.path.join(dest_dir, "core-site.xml"), "w") as f:
        f.write(core_site_xml_content)

In [6]:
hive_site_xml_content = f"""<?xml version="1.0"?>
<configuration>
    <property>
        <name>hive.users.in.admin.role</name>
        <value>hive,root,hadoop</value>
    </property>
    <property>
        <name>hive.server2.thrift.bind.host</name>
        <value>0.0.0.0</value>
    </property>
    <property>
        <name>hive.server2.webui.host</name>
        <value>0.0.0.0</value>
    </property>

    <property>
        <name>javax.jdo.option.ConnectionURL</name>
        <value>jdbc:postgresql://{HIVE_DB_HOSTNAME}:{HIVE_DB_EXTERNAL_PORT}/{POSTGRES_DB}</value>
    </property>
    <property>
        <name>javax.jdo.option.ConnectionDriverName</name>
        <value>org.postgresql.Driver</value>
    </property>
    <property>
        <name>javax.jdo.option.ConnectionUserName</name>
        <value>{POSTGRES_USER}</value>
    </property>
    <property>
        <name>javax.jdo.option.ConnectionPassword</name>
        <value>{POSTGRES_PASSWORD}</value>
    </property>

    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}</value>
    </property>
    <property>
        <name>dfs.datanode.use.datanode.hostname</name>
        <value>true</value>
    </property>
    <property>
        <name>hive.metastore.uris</name>
        <value>thrift://{HIVE_METASTORE_HOSTNAME}:{HIVE_METASTORE_EXTERNAL_PORT}</value>
    </property>
    <property>
        <name>yarn.resourcemanager.address</name>
        <value>{HADOOP_RESOURCEMANAGER_HOSTNAME}:{HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT}</value>
    </property>
    <property>
        <name>yarn.resourcemanager.scheduler.address</name>
        <value>{HADOOP_RESOURCEMANAGER_HOSTNAME}:{HADOOP_RESOURCEMANAGER_SCHEDULER_PORT}</value>
    </property>
    <property>
        <name>mapreduce.framework.name</name>
        <value>yarn</value>
    </property>

    <property>
        <name>hive.server2.enable.doAs</name>
        <value>false</value>
    </property>
    <property>
        <name>hive.server2.thrift.port</name>
        <value>10000</value>
    </property>
    <property>
        <name>hive.server2.transport.mode</name>
        <value>binary</value>
    </property>
    <property>
        <name>hive.plan.serialization.format</name>
        <value>javaXML</value>
    </property>
    <property>
        <name>hive.exec.submit.local.task.via.child</name>
        <value>false</value>
    </property>

    <property>
        <name>hive.metastore.event.db.listener.timetolive</name>
        <value>0s</value>
    </property>
    <property>
        <name>hive.notification.event.poll.interval</name>
        <value>0s</value>
    </property>

    <property>
        <name>tez.lib.uris</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/apps/dist-lib/dist-lib.tar.gz</value>
    </property>
    <property>
        <name>tez.use.cluster.hadoop-libs</name>
        <value>true</value> 
    </property>
    <property>
        <name>tez.ignore.lib.uris</name>
        <value>false</value>
    </property>
    <property>
        <name>hive.execution.engine</name>
        <value>tez</value> 
    </property>
    <property>
        <name>tez.local.mode</name>
        <value>false</value>
    </property>
    <property>
        <name>tez.runtime.optimize.local.fetch</name>
        <value>true</value>
    </property>

    <property>
        <name>hive.exec.scratchdir</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/hive/scratch</value>
    </property>
    <property>
        <name>hive.exec.stagingdir</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/hive/staging</value>
    </property>
    <property>
        <name>hive.exec.local.scratchdir</name>
        <value>/tmp/hive/local</value>
    </property>
    <property>
        <name>hive.start.cleanup.scratchdir</name>
        <value>true</value>
    </property>
    <property>
        <name>hive.fetch.task.conversion</name>
        <value>more</value>
    </property>
    <property>
        <name>hive.exec.mode.local.auto</name>
        <value>true</value>
    </property>
</configuration>
"""

for container in [HIVE_METASTORE_CONTAINER_NAME, HIVE_SERVER2_CONTAINER_NAME]:
    dest_dir = os.path.join(DOCKER_MOUNTDIR, container, "hive_custom_conf")
    os.makedirs(dest_dir, exist_ok=True)
    with open(os.path.join(dest_dir, "hive-site.xml"), "w") as f:
        f.write(hive_site_xml_content)

In [7]:
tez_site_content = f"""<?xml version="1.0"?>
<configuration>
    <property>
        <name>tez.lib.uris</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/apps/dist-lib/dist-lib.tar.gz</value>
    </property>
    <property>
        <name>tez.use.cluster.hadoop-libs</name>
        <value>true</value> 
    </property>
    <property>
        <name>tez.ignore.lib.uris</name>
        <value>false</value>
    </property>
    <property>
        <name>tez.local.mode</name>
        <value>false</value>
    </property>
    <property>
        <name>tez.runtime.optimize.local.fetch</name>
        <value>true</value>
    </property>
    <property>
        <name>tez.cluster.additional.classpath.prefix</name>
        <value>/opt/hadoop/share/hadoop/common/*:/opt/hadoop/share/hadoop/common/lib/*:/opt/hadoop/share/hadoop/hdfs/*:/opt/hadoop/share/hadoop/hdfs/lib/*:/opt/hadoop/share/hadoop/yarn/*:/opt/hadoop/share/hadoop/yarn/lib/*:/opt/hive/lib/*:/opt/hive/conf</value>
    </property>
    <property>
        <name>tez.am.hostname</name>
        <value>0.0.0.0</value> 
    </property>
    <property>
        <name>tez.am.launch.cmd-opts</name>
        <value>-Djava.net.preferIPv4Stack=true -Dhadoop.rpc.protection=authentication</value>
    </property>

    <property>
        <name>hive.exec.scratchdir</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/hive/scratch</value>
    </property>
    <property>
        <name>hive.exec.stagingdir</name>
        <value>hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}/tmp/hive/staging</value>
    </property>
    <property>
        <name>hive.exec.local.scratchdir</name>
        <value>/tmp/hive/local</value>
    </property>
    <property>
        <name>hive.start.cleanup.scratchdir</name>
        <value>true</value>
    </property>
    
    <property>
        <name>tez.history.logging.service.class</name>
        <value>org.apache.tez.dag.history.logging.ats.ATSHistoryLoggingService</value>
    </property>

    <property>
        <name>mapreduce.framework.name</name>
        <value>yarn</value>
    </property>
    <property>
        <name>hive.exec.submit.local.task.via.child</name>
        <value>false</value>
    </property>
</configuration>
"""

for container in [HIVE_METASTORE_CONTAINER_NAME, HIVE_SERVER2_CONTAINER_NAME]:
    dest_dir = os.path.join(DOCKER_MOUNTDIR, container, "hive_custom_conf")
    os.makedirs(dest_dir, exist_ok=True)
    with open(os.path.join(dest_dir, "tez-site.xml"), "w") as f:
        f.write(tez_site_content)

In [8]:
import os

# 1. Definimos el contenido del archivo de logs
log4j_properties_content = """#
log4j.rootLogger=INFO, console

# Configuración del appender (salida estándar)
log4j.appender.console=org.apache.log4j.ConsoleAppender
log4j.appender.console.target=System.err
log4j.appender.console.layout=org.apache.log4j.PatternLayout
log4j.appender.console.layout.ConversionPattern=%d{yy/MM/dd HH:mm:ss} %p %c{2}: %m%n

# Silenciar componentes específicos que son muy ruidosos
log4j.logger.org.apache.hadoop=INFO
log4j.logger.org.apache.hive=INFO
log4j.logger.org.apache.tez=INFO
log4j.logger.org.apache.parquet=INFO
log4j.logger.org.apache.iceberg=INFO
"""

# 2. Iteramos sobre los contenedores para escribir el archivo
for container in [HIVE_METASTORE_CONTAINER_NAME, HIVE_SERVER2_CONTAINER_NAME]:
    dest_dir = os.path.join(DOCKER_MOUNTDIR, container, "hive_custom_conf")
    os.makedirs(dest_dir, exist_ok=True)
    with open(os.path.join(dest_dir, "log4j.properties"), "w") as f:
        f.write(log4j_properties_content)

# Start hive-cluster.docker-compose.yml

In [9]:
import yaml
from IPython.display import Markdown, display

compose_filename = os.path.join(LOCAL_WORKDIR, "hive-cluster.docker-compose.yml")

# Usamos EXTERNAL_PORT para respetar tu VPN/DNS
metastore_jdbc_opts = f" ".join(
    [
        f"-Dhive.metastore.db.type=postgres",
        f"-Dmetastore.db.type=postgres",
        f"-Djavax.jdo.option.ConnectionDriverName=org.postgresql.Driver",
        f"-Djavax.jdo.option.ConnectionURL=jdbc:postgresql://{HIVE_DB_HOSTNAME}:{HIVE_DB_EXTERNAL_PORT}/{POSTGRES_DB}",
        f"-Djavax.jdo.option.ConnectionUserName={POSTGRES_USER}",
        f"-Djavax.jdo.option.ConnectionPassword={POSTGRES_PASSWORD}",
    ]
)

hdfs_opts = f" ".join(
    [
        f"-Dfs.defaultFS=hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}",
        f"-Dhive.metastore.warehouse.dir=hdfs://{HADOOP_NAMENODE_HOSTNAME}:{HADOOP_NAMENODE_PORT}{HIVE_DATADIR}/managed",
        "-Dmapreduce.framework.name=yarn",
        f"-Dyarn.resourcemanager.address={HADOOP_RESOURCEMANAGER_HOSTNAME}:{HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT}",
        f"-Dmapreduce.application.classpath=/opt/hadoop/share/hadoop/mapreduce/*:/opt/hadoop/share/hadoop/mapreduce/lib/*:/opt/hadoop/share/hadoop/common/*:/opt/hadoop/share/hadoop/common/lib/*:/opt/hadoop/share/hadoop/hdfs/*:/opt/hadoop/share/hadoop/hdfs/lib/*:/opt/hadoop/share/hadoop/yarn/*:/opt/hadoop/share/hadoop/yarn/lib/*",
    ]
)

hive_compose_dict = {
    "name": "hive-cluster",
    # "networks": {"hive-cluster": {"driver": "bridge"}},
    # "networks": {
    #     "hadoop-cluster": {"external": True, "name": "hadoop-cluster_hadoop-cluster"}
    # },
    "networks": {
        "hadoop-cluster": {
            "external": True,
            "name": "hadoop-cluster_hadoop-cluster",
        }
    },
    "services": {
        HIVE_DB_CONTAINER_NAME: {
            "image": POSTGRES_IMAGE,
            "container_name": HIVE_DB_CONTAINER_NAME,
            "hostname": HIVE_DB_HOSTNAME,
            "environment": {
                "POSTGRES_USER": POSTGRES_USER,
                "POSTGRES_PASSWORD": POSTGRES_PASSWORD,
                "POSTGRES_DB": POSTGRES_DB,
                "PGPORT": str(HIVE_DB_INTERNAL_PORT),
            },
            "volumes": [f".\\mount\\{HIVE_DB_CONTAINER_NAME}:/var/lib/postgresql"],
            "networks": ["hadoop-cluster"],
            "ports": [f"{HIVE_DB_EXTERNAL_PORT}:{HIVE_DB_INTERNAL_PORT}"],
            "extra_hosts": [f"{DOCKER_INTERNAL_HOST}:host-gateway"],
            "dns": DOCKER_DNS,
            "deploy": {"resources": {"limits": {"cpus": "1.0", "memory": "512M"}}},
            "healthcheck": {
                "test": [
                    "CMD-SHELL",
                    f"pg_isready -h 127.0.0.1 -p {HIVE_DB_INTERNAL_PORT} -U {POSTGRES_USER} -d {POSTGRES_DB}",
                ],
                "interval": "5s",
                "timeout": "5s",
                "retries": 20,
                "start_period": "15s",
            },
        },
        HIVE_SCHEMA_INIT_CONTAINER_NAME: {
            "image": APACHE_HIVE_IMAGE,
            "container_name": HIVE_SCHEMA_INIT_CONTAINER_NAME,
            "hostname": HIVE_SCHEMA_INIT_HOSTNAME,
            "depends_on": {HIVE_DB_CONTAINER_NAME: {"condition": "service_healthy"}},
            "environment": {
                "DB_DRIVER": "postgres",
                "SERVICE_OPTS": f"{metastore_jdbc_opts}",
                # "HADOOP_CLASSPATH": "/hive_custom_conf:/opt/hive/lib/postgresql-42.7.10.jar:/opt/hive/lib/*:/opt/tez/*:/opt/tez/lib/*",
                "HADOOP_CLASSPATH": "/hive_custom_conf:/opt/hive/lib/postgresql-42.7.10.jar",
            },
            "volumes": [
                "./postgresql-42.7.10.jar:/opt/hive/lib/postgresql-42.7.10.jar"
            ],
            "networks": ["hadoop-cluster"],
            "extra_hosts": [f"{DOCKER_INTERNAL_HOST}:host-gateway"],
            "dns": DOCKER_DNS,
            "deploy": {"resources": {"limits": {"cpus": "2.0", "memory": "512M"}}},
        },
        HIVE_METASTORE_CONTAINER_NAME: {
            "image": APACHE_HIVE_IMAGE,
            "container_name": HIVE_METASTORE_CONTAINER_NAME,
            "hostname": HIVE_METASTORE_HOSTNAME,
            "depends_on": {
                HIVE_SCHEMA_INIT_CONTAINER_NAME: {
                    "condition": "service_completed_successfully"
                }
            },
            "environment": {
                "DB_DRIVER": "postgres",
                "IS_RESUME": "true",
                "SERVICE_NAME": "metastore",
                "HIVE_CUSTOM_CONF_DIR": "/hive_custom_conf",
                "TEZ_CONF_DIR": "/hive_custom_conf",
                "HADOOP_CONF_DIR": "/hive_custom_conf",
                "SERVICE_OPTS": f"{metastore_jdbc_opts} {hdfs_opts}",
                # "HADOOP_CLASSPATH": "/hive_custom_conf:/opt/hive/lib/postgresql-42.7.10.jar:/opt/hive/lib/*:/opt/tez/*:/opt/tez/lib/*",
                "HADOOP_CLASSPATH": "/hive_custom_conf:/opt/hive/lib/postgresql-42.7.10.jar:/opt/tez/*:/opt/tez/lib/*",
            },
            "volumes": [
                "./postgresql-42.7.10.jar:/opt/hive/lib/postgresql-42.7.10.jar",
                f"./mount/{HIVE_METASTORE_CONTAINER_NAME}/hive_custom_conf:/hive_custom_conf",
                f"./mount/{HIVE_METASTORE_CONTAINER_NAME}/hive_custom_conf/core-site.xml:/opt/hadoop/etc/hadoop/core-site.xml",
            ],
            "networks": ["hadoop-cluster"],
            "ports": [f"{HIVE_METASTORE_EXTERNAL_PORT}:{HIVE_METASTORE_INTERNAL_PORT}"],
            "extra_hosts": [f"{DOCKER_INTERNAL_HOST}:host-gateway"],
            "dns": DOCKER_DNS,
            "deploy": {"resources": {"limits": {"cpus": "1.0", "memory": "512M"}}},
            "healthcheck": {
                "test": [
                    "CMD-SHELL",
                    f"bash -c '</dev/tcp/127.0.0.1/{HIVE_METASTORE_INTERNAL_PORT}'",
                ],
                "interval": "5s",
                "timeout": "5s",
                "retries": 20,
                "start_period": "15s",
            },
        },
        HIVE_SERVER2_CONTAINER_NAME: {
            "image": APACHE_HIVE_IMAGE,
            "container_name": HIVE_SERVER2_CONTAINER_NAME,
            "hostname": HIVE_SERVER2_HOSTNAME,
            "depends_on": {
                HIVE_METASTORE_CONTAINER_NAME: {"condition": "service_healthy"}
            },
            "environment": {
                "DB_DRIVER": "postgres",
                "IS_RESUME": "true",
                "SERVICE_NAME": "hiveserver2",
                "HIVE_CUSTOM_CONF_DIR": "/hive_custom_conf",
                "TEZ_CONF_DIR": "/hive_custom_conf",
                "HADOOP_CONF_DIR": "/hive_custom_conf",
                "SERVICE_OPTS": f"-Dhive.metastore.uris=thrift://{HIVE_METASTORE_HOSTNAME}:{HIVE_METASTORE_EXTERNAL_PORT} {hdfs_opts}",
                # "HADOOP_CLASSPATH": "/hive_custom_conf:/opt/hive/lib/postgresql-42.7.10.jar:/opt/hive/lib/*:/opt/tez/*:/opt/tez/lib/*",
                "HADOOP_CLASSPATH": "/hive_custom_conf:/opt/hive/lib/postgresql-42.7.10.jar:/opt/tez/*:/opt/tez/lib/*",
            },
            "volumes": [
                "./postgresql-42.7.10.jar:/opt/hive/lib/postgresql-42.7.10.jar",
                f"./mount/{HIVE_SERVER2_CONTAINER_NAME}/hive_custom_conf:/hive_custom_conf",
                f"./mount/{HIVE_SERVER2_CONTAINER_NAME}/hive_custom_conf/core-site.xml:/opt/hadoop/etc/hadoop/core-site.xml",
            ],
            "networks": ["hadoop-cluster"],
            "ports": [
                f"{HIVE_SERVER2_EXTERNAL_PORT}:{HIVE_SERVER2_INTERNAL_PORT}",
                f"{HIVE_SERVER2_UI_EXTERNAL_PORT}:{HIVE_SERVER2_UI_INTERNAL_PORT}",
            ],
            "extra_hosts": [f"{DOCKER_INTERNAL_HOST}:host-gateway"],
            "dns": DOCKER_DNS,
            "deploy": {"resources": {"limits": {"cpus": "1.0", "memory": "768M"}}},
            "healthcheck": {
                "test": [
                    "CMD-SHELL",
                    f"bash -c '</dev/tcp/127.0.0.1/{HIVE_SERVER2_INTERNAL_PORT}'",
                ],
                "interval": "5s",
                "timeout": "5s",
                "retries": 30,
                "start_period": "15s",
            },
        },
    },
}

yaml_contents = yaml.dump(
    hive_compose_dict, default_flow_style=False, sort_keys=False, indent=4
)
with open(compose_filename, "w") as f:
    f.write(yaml_contents)

In [10]:
# !docker exec namenode hdfs dfsadmin -safemode leave

In [11]:
if HIVE_START_FROM_SCRATCH:
    !docker exec -u hadoop namenode hdfs dfs -rm -r -f -skipTrash /tmp/hive/*
    !docker exec -u hadoop namenode hdfs dfs -rm -r -f -skipTrash {HIVE_DATADIR}/managed/*
    !docker exec -u hadoop namenode hdfs dfs -rm -r -f -skipTrash {HIVE_DATADIR}/external/*

!docker exec -u root namenode groupadd hive
!docker exec -u root namenode useradd -m -g hive hive
!docker exec -u root namenode usermod -aG sudo hive
!docker exec -u root namenode usermod -aG hadoop hive
!docker exec -u root namenode bash -c "grep -qFx 'hive ALL=(ALL) NOPASSWD:ALL' /etc/sudoers || echo 'hive ALL=(ALL) NOPASSWD:ALL' >> /etc/sudoers"
# !docker exec -u root namenode bash -c "echo 'hive ALL=(ALL) NOPASSWD:ALL' >> /etc/sudoers"
!docker exec -u hadoop namenode hdfs dfs -mkdir -p /tmp/hive
!docker exec -u hadoop namenode hdfs dfs -chown -R hive:hive /tmp/hive
!docker exec -u hadoop namenode hdfs dfs -chmod -R 777 /tmp
!docker exec -u hadoop namenode hdfs dfs -mkdir -p {HIVE_DATADIR}/managed
!docker exec -u hadoop namenode hdfs dfs -mkdir -p {HIVE_DATADIR}/external
!docker exec -u hadoop namenode hdfs dfs -chown -R hive:hive {HIVE_USERDIR}
!docker exec -u hadoop namenode hdfs dfs -chmod -R 777 {HIVE_USERDIR}

zsh:1: no matches found: /tmp/hive/*
zsh:1: no matches found: /user/hive/warehouse/managed/*
zsh:1: no matches found: /user/hive/warehouse/external/*


In [12]:
!docker compose -f hive-cluster.docker-compose.yml up -d --wait

[+] up 1/2
 ✔ Container hive-metastore-db Created                                      0.0s
 ⠋ Container hive-schema-init  Creating                                     0.0s
[+] up 4/4
 ✔ Container hive-metastore-db Created                                      0.0s
 ✔ Container hive-schema-init  Created                                      0.0s
 ✔ Container hive-metastore    Created                                      0.0s
 ✔ Container hive-server2      Created                                      0.0s
[+] up 3/4
 ⠙ Container hive-metastore-db Waiting                                      0.3s
 ✔ Container hive-schema-init  Created                                      0.0s
 ✔ Container hive-metastore    Created                                      0.0s
 ✔ Container hive-server2      Created                                      0.0s
[+] up 3/4
 ⠹ Container hive-metastore-db Waiting                                      0.4s
 ✔ Container hive-schema-init  Created                           

# Add .beeline dir to hive $HOME

In [13]:
!docker exec -u root hive-server2 bash -c "mkdir -p /home/hive/.beeline && chown -R hive:hive /home/hive"

# Load dist-lib (tez, common-collections) to HDFS

In [14]:
import os

print("--- CONSOLIDANDO Y EMPAQUETANDO DISTRIBUTED CACHE LIB ---")

# Script que se ejecutará DENTRO de hive-server2 para armar el paquete unificado
build_tar_script = """#!/bin/bash
set -e

echo "1. Creando directorio unificado /tmp/dist-lib..."
rm -rf /tmp/dist-lib/* /tmp/dist-lib.tar.gz
mkdir -p /tmp/dist-lib/conf
mkdir -p /tmp/dist-lib/lib

echo "2. Copiando Tez nativo..."
cp -r /opt/tez/* /tmp/dist-lib/

echo "3. Inyectando commons-collections y commons-lang de Hive..."
cp /opt/hive/lib/commons-collections-*.jar /opt/hive/lib/commons-lang-*.jar /tmp/dist-lib/lib/ 2>/dev/null || true

echo "4. Generando log4j seguro..."
cat << 'EOF' > /tmp/dist-lib/conf/tez-container-log4j.properties
tez.root.logger=INFO,CLA
log4j.rootLogger=${tez.root.logger}
log4j.appender.CLA=org.apache.log4j.ConsoleAppender
log4j.appender.CLA.Target=System.out
log4j.appender.CLA.layout=org.apache.log4j.PatternLayout
log4j.appender.CLA.layout.ConversionPattern=%d{ISO8601} [%p] [%t] |%c{2}|: %m%n
EOF

echo "5. Eliminando SLF4J duplicados para evitar conflictos..."
rm -f /tmp/dist-lib/lib/slf4j-*.jar

echo "6. Comprimiendo el paquete final..."
cd /tmp/dist-lib
# El asterisco asegura que no haya carpetas intermedias, tal como YARN lo espera
tar -czf /tmp/dist-lib.tar.gz *

echo "Paquete construido con éxito."
"""

with open("build_dist_lib_tar.sh", "w", newline='\n') as f:
    f.write(build_tar_script)

# Ejecutamos la construcción dentro de hive-server2
print("Construyendo el tarball dentro de hive-server2...")
!docker cp build_dist_lib_tar.sh hive-server2:/tmp/build_dist_lib_tar.sh
!docker exec -u root hive-server2 bash /tmp/build_dist_lib_tar.sh

# Extraemos SOLO el tarball ya terminado y lo pasamos al namenode
print("Transfiriendo el tarball terminado al Namenode...")
!docker cp hive-server2:/tmp/dist-lib.tar.gz ./dist-lib.tar.gz
!docker cp ./dist-lib.tar.gz namenode:/tmp/dist-lib.tar.gz

# Subimos a HDFS desde el namenode a la nueva ruta
print("Subiendo a HDFS en /apps/dist-lib ...")

!docker exec namenode hdfs dfs -mkdir -p /apps/dist-lib
!docker exec namenode hdfs dfs -rm -r -f /apps/dist-lib/*
!docker exec namenode hdfs dfs -put /tmp/dist-lib.tar.gz /apps/dist-lib/dist-lib.tar.gz
!docker exec namenode hdfs dfs -chmod -R 777 /apps/dist-lib

print("✅ ¡ÉXITO! Distributed Cache unificado y subido a HDFS en /apps/dist-lib.")

# 4. Limpieza ultra rápida
os.remove("./dist-lib.tar.gz")
os.remove("./build_dist_lib_tar.sh")
print("Archivos temporales locales limpios.")

--- CONSOLIDANDO Y EMPAQUETANDO DISTRIBUTED CACHE LIB ---
Construyendo el tarball dentro de hive-server2...
Successfully copied 3.07kB to hive-server2:/tmp/build_dist_lib_tar.sh
1. Creando directorio unificado /tmp/dist-lib...
2. Copiando Tez nativo...
3. Inyectando commons-collections y commons-lang de Hive...
4. Generando log4j seguro...
5. Eliminando SLF4J duplicados para evitar conflictos...
6. Comprimiendo el paquete final...
Paquete construido con éxito.
Transfiriendo el tarball terminado al Namenode...
Successfully copied 19.8MB to /Users/leonardovillarrealcerda/Documents/ITAM/ITAM_Semestre6/bases_no_rel/hadoop/dist-lib.tar.gz
Successfully copied 19.8MB to namenode:/tmp/dist-lib.tar.gz
Subiendo a HDFS en /apps/dist-lib ...
zsh:1: no matches found: /apps/dist-lib/*
✅ ¡ÉXITO! Distributed Cache unificado y subido a HDFS en /apps/dist-lib.
Archivos temporales locales limpios.
